# Estudo de Caso 6.2 — Curva de Retenção de van Genuchten

**Capítulo Associado:** Capítulo 6 — Percolação em Meio Poroso  
**Nível:** Graduação  
**Tempo estimado:** 45 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, exploramos o modelo de van Genuchten-Mualem para diferentes texturas de solo:

1. **Comparar** curvas de retenção $\theta(\psi)$ para areia, franco e argila
2. **Analisar** a condutividade hidráulica $K(\theta)$ em escala logarítmica
3. **Explorar** o efeito dos parâmetros $\alpha$, $n$ e $m$
4. **Estimar** parâmetros usando funções de pedotransferência (Rosetta)

---

## 🎯 Objetivos de Aprendizagem

- Implementar o modelo de van Genuchten-Mualem
- Interpretar fisicamente os parâmetros $\theta_r$, $\theta_s$, $\alpha$, $n$
- Comparar comportamentos hidráulicos de diferentes texturas
- Entender a relação entre curva de retenção e condutividade

---

## 🔧 Requisitos

- Bibliotecas: `numpy`, `matplotlib`
- Conhecimento prévio: Modelo de van Genuchten (Cap. 6)

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ Ambiente configurado com sucesso!")

---

## 📊 Seção 1: Parâmetros de Diferentes Texturas

### Tabela de Parâmetros (Rosetta)

| Classe Textural | Areia (%) | Silte (%) | Argila (%) | $\theta_r$ | $\theta_s$ | $\alpha$ [m⁻¹] | $n$ | $K_{sat}$ [m/dia] |
|-----------------|-----------|-----------|------------|------------|------------|----------------|-----|-------------------|
| Areia | 90 | 5 | 5 | 0,045 | 0,43 | 14,5 | 2,68 | 29,7 |
| Franco-arenosa | 70 | 20 | 10 | 0,058 | 0,41 | 7,4 | 1,56 | 4,4 |
| Franco | 40 | 40 | 20 | 0,068 | 0,43 | 2,1 | 1,41 | 1,3 |
| Franco-argilosa | 30 | 30 | 40 | 0,075 | 0,45 | 0,8 | 1,24 | 0,3 |
| Argila | 10 | 30 | 60 | 0,082 | 0,48 | 0,4 | 1,15 | 0,05 |

In [ ]:
# ============================================================
# PARÂMETROS DE DIFERENTES TEXTURAS
# ============================================================

solos = {
    'Areia': {
        'theta_r': 0.045, 'theta_s': 0.43, 'alpha': 14.5, 'n': 2.68,
        'K_sat': 29.7/86400,  # m/dia -> m/s
        'textura': '90% areia, 5% silte, 5% argila'
    },
    'Franco-arenosa': {
        'theta_r': 0.058, 'theta_s': 0.41, 'alpha': 7.4, 'n': 1.56,
        'K_sat': 4.4/86400,
        'textura': '70% areia, 20% silte, 10% argila'
    },
    'Franco': {
        'theta_r': 0.068, 'theta_s': 0.43, 'alpha': 2.1, 'n': 1.41,
        'K_sat': 1.3/86400,
        'textura': '40% areia, 40% silte, 20% argila'
    },
    'Franco-argilosa': {
        'theta_r': 0.075, 'theta_s': 0.45, 'alpha': 0.8, 'n': 1.24,
        'K_sat': 0.3/86400,
        'textura': '30% areia, 30% silte, 40% argila'
    },
    'Argila': {
        'theta_r': 0.082, 'theta_s': 0.48, 'alpha': 0.4, 'n': 1.15,
        'K_sat': 0.05/86400,
        'textura': '10% areia, 30% silte, 60% argila'
    }
}

print("=" * 60)
print("PARÂMETROS DE VAN GENUCHTEN POR TEXTURA")
print("=" * 60)

print(f"\n{'Solo':<18} {'θ_r':<8} {'θ_s':<8} {'α':<8} {'n':<8} {'K_sat [m/dia]':<15}")
print("-" * 65)
for nome, params in solos.items():
    print(f"{nome:<18} {params['theta_r']:<8.3f} {params['theta_s']:<8.2f} "
          f"{params['alpha']:<8.1f} {params['n']:<8.2f} {params['K_sat']*86400:<15.1f}")

---

## 🧮 Seção 2: Modelo de van Genuchten-Mualem

In [ ]:
# ============================================================
# FUNÇÕES DE VAN GENUCHTEN
# ============================================================

def van_genuchten_theta(psi, theta_r, theta_s, alpha, n):
    """Conteúdo de água θ em função do potencial matricial ψ."""
    m = 1 - 1/n
    psi = np.atleast_1d(psi)
    theta = np.zeros_like(psi)
    
    # Para ψ >= 0 (saturado)
    theta[psi >= 0] = theta_s
    
    # Para ψ < 0 (não-saturado)
    mask = psi < 0
    psi_neg = np.abs(psi[mask])
    theta[mask] = theta_r + (theta_s - theta_r) / (1 + (alpha * psi_neg)**n)**m
    
    return theta

def van_genuchten_K(theta, theta_r, theta_s, K_sat, n, L=0.5):
    """Condutividade hidráulica K em função do conteúdo de água θ."""
    m = 1 - 1/n
    theta = np.atleast_1d(theta)
    Se = (theta - theta_r) / (theta_s - theta_r)
    Se = np.clip(Se, 0, 1)
    
    K = K_sat * Se**L * (1 - (1 - Se**(1/m))**m)**2
    
    return K

print("✅ Funções de van Genuchten implementadas!")

---

## 📈 Seção 3: Curvas de Retenção θ(ψ)

### Comparação entre Diferentes Texturas

In [ ]:
# ============================================================
# CURVAS DE RETENÇÃO θ(ψ)
# ============================================================

print("=" * 60)
print("CURVAS DE RETENÇÃO θ(ψ)")
print("=" * 60)

# Gerar curvas
psi_range = np.logspace(-4, 2, 200)  # de 0,0001 a 100 m (valores absolutos)
psi_range = -psi_range  # Tornar negativo (sucção)

fig, ax = plt.subplots(figsize=(12, 8))

cores = ['blue', 'green', 'orange', 'red', 'purple']
markers = ['o', 's', '^', 'D', 'v']

for i, (nome, params) in enumerate(solos.items()):
    theta_range = van_genuchten_theta(
        psi_range, 
        params['theta_r'], 
        params['theta_s'], 
        params['alpha'], 
        params['n']
    )
    
    ax.plot(np.abs(psi_range), theta_range, 
            color=cores[i], linewidth=2, 
            label=f"{nome} ({params['textura']})")

ax.set_xlabel('|ψ| [m] (Sucção)', fontsize=13)
ax.set_ylabel('θ [m³/m³]', fontsize=13)
ax.set_title('Curvas de Retenção de van Genuchten', fontsize=15, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc='upper right')

# Adicionar linhas de referência
ax.axhline(y=0.4, color='gray', linestyle=':', alpha=0.5)
ax.text(0.001, 0.41, 'θ ≈ 0,4', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print("\n💡 Interpretação:")
print("  • Areia: curva mais íngreme (drena rapidamente)")
print("  • Argila: curva mais suave (retém mais água)")
print("  • Franco: comportamento intermediário")

---

## 📊 Seção 4: Condutividade Hidráulica K(θ)

### Comparação em Escala Logarítmica

In [ ]:
# ============================================================
# CONDUTIVIDADE HIDRÁULICA K(θ)
# ============================================================

print("=" * 60)
print("CONDUTIVIDADE HIDRÁULICA K(θ)")
print("=" * 60)

fig, ax = plt.subplots(figsize=(12, 8))

for i, (nome, params) in enumerate(solos.items()):
    # Gerar curva K(θ)
    theta_range = np.linspace(params['theta_r'] + 0.001, params['theta_s'], 100)
    K_range = van_genuchten_K(
        theta_range, 
        params['theta_r'], 
        params['theta_s'], 
        params['K_sat'], 
        params['n']
    )
    
    ax.plot(theta_range, K_range * 86400,  # Converter para m/dia
            color=cores[i], linewidth=2, 
            label=f"{nome}")

ax.set_xlabel('θ [m³/m³]', fontsize=13)
ax.set_ylabel('K [m/dia]', fontsize=13)
ax.set_title('Condutividade Hidráulica Não-Saturada', fontsize=15, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\n💡 Interpretação:")
print("  • K varia exponencialmente com θ (várias ordens de grandeza)")
print("  • Areia: K alto mesmo para θ moderado")
print("  • Argila: K muito baixo (impermeável) para θ < 0,3")
print("  • Essa não-linearidade exige solução numérica da equação de Richards")

---

## 🔍 Seção 5: Efeito dos Parâmetros α e n

### Análise de Sensibilidade

In [ ]:
# ============================================================
# EFEITO DOS PARÂMETROS α E n
# ============================================================

print("=" * 60)
print("EFEITO DOS PARÂMETROS α E n")
print("=" * 60)

# Parâmetros base (Franco-arenosa)
theta_r_base = 0.058
theta_s_base = 0.41
alpha_base = 7.4
n_base = 1.56

# Variar α
alpha_values = [2.0, 7.4, 15.0, 30.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Efeito de α
for alpha_teste in alpha_values:
    theta_teste = van_genuchten_theta(
        psi_range, theta_r_base, theta_s_base, alpha_teste, n_base
    )
    axes[0].plot(np.abs(psi_range), theta_teste, linewidth=2, 
                 label=f'α = {alpha_teste}')

axes[0].set_xlabel('|ψ| [m]', fontsize=12)
axes[0].set_ylabel('θ [m³/m³]', fontsize=12)
axes[0].set_title('Efeito do Parâmetro α', fontsize=14, fontweight='bold')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)

# Variar n
n_values = [1.2, 1.56, 2.0, 3.0]

# Gráfico 2: Efeito de n
for n_teste in n_values:
    theta_teste = van_genuchten_theta(
        psi_range, theta_r_base, theta_s_base, alpha_base, n_teste
    )
    axes[1].plot(np.abs(psi_range), theta_teste, linewidth=2, 
                 label=f'n = {n_teste}')

axes[1].set_xlabel('|ψ| [m]', fontsize=12)
axes[1].set_ylabel('θ [m³/m³]', fontsize=12)
axes[1].set_title('Efeito do Parâmetro n', fontsize=14, fontweight='bold')
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\n💡 Interpretação:")
print("  • α controla a posição da curva (sucção de entrada de ar)")
print("    - α alto: solo drena em sucções baixas (areia)")
print("    - α baixo: solo retém água em sucções altas (argila)")
print("  • n controla a inclinação da curva")
print("    - n alto: transição abrupta (solo uniforme)")
print("    - n baixo: transição suave (solo heterogêneo)")

---

## 📋 Seção 6: Tabela Comparativa

### Características Hidráulicas por Textura

In [ ]:
# ============================================================
# TABELA COMPARATIVA
# ============================================================

print("=" * 60)
print("CARACTERÍSTICAS HIDRÁULICAS POR TEXTURA")
print("=" * 60)

print(f"\n{'Solo':<18} {'Drenagem':<12} {'Retenção':<12} {'Aplicação':<20}")
print("-" * 62)

caracteristicas = {
    'Areia': ('Muito rápida', 'Baixa', 'Drenagem, fundações'),
    'Franco-arenosa': ('Rápida', 'Moderada', 'Agricultura, pastagem'),
    'Franco': ('Moderada', 'Boa', 'Agricultura intensiva'),
    'Franco-argilosa': ('Lenta', 'Muito boa', 'Culturas exigentes'),
    'Argila': ('Muito lenta', 'Excelente', 'Impermeabilização')
}

for nome, (drenagem, retencao, aplicacao) in caracteristicas.items():
    print(f"{nome:<18} {drenagem:<12} {retencao:<12} {aplicacao:<20}")

print("\n💡 Interpretação:")
print("  • Solos arenosos: alta permeabilidade, baixa capacidade de retenção")
print("  • Solos argilosos: baixa permeabilidade, alta capacidade de retenção")
print("  • Solos francos: equilíbrio ideal para agricultura")

---

## 💡 Seção 7: Discussão e Conclusões

### Resultados Principais

1. **Curvas de retenção:** O modelo de van Genuchten captura adequadamente o comportamento sigmoidal da relação $\theta(\psi)$ para diferentes texturas.

2. **Condutividade hidráulica:** $K(\theta)$ varia exponencialmente com o conteúdo de água, refletindo a não-linearidade do transporte em meio não-saturado.

3. **Parâmetros α e n:**
   - $\alpha$ controla a sucção de entrada de ar (posição da curva)
   - $n$ controla a inclinação da curva (uniformidade do solo)

### Aplicações Práticas

- **Agricultura:** Seleção de culturas conforme a capacidade de retenção do solo
- **Geotecnia:** Estabilidade de taludes e barragens de terra
- **Hidrologia:** Modelagem de infiltração e recarga de aquíferos
- **Meio ambiente:** Projeto de barreiras impermeáveis em aterros

### Limitações do Modelo

- **Histerese:** O modelo não captura efeitos de histerese (curvas de secagem e umedecimento diferentes)
- **Estrutura do solo:** Assume solo homogêneo, não considerando macroporos ou fraturas
- **Dependência temporal:** Não considera efeitos de compactação ou swelling

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 6](../notebooks/06_Percolacao_Meio_Poroso.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 6.2**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>